# Sheet 9 — Training a neural network

In this sheet you will build and train a small feed-forward neural network from scratch, using nothing but `numpy`. Everything runs in a second on a laptop. The goal is to see that the machinery of Section 7 of the notes (the forward pass, backpropagation, gradient descent) is just a few matrix products you can write yourself.

The task is one-dimensional regression. You are given a dataset of pairs $(x_i, y_i)$ in `data/sheet_9_data.txt` drawn from some unknown process. Your job is to learn the mapping from the data alone. We use a network with a single hidden layer of width $H$, $\tanh$ activation, and a linear output (Eqs. 7.22–7.23):
$$
\boldsymbol{z} = \boldsymbol{W}^{(1)} x + \boldsymbol{b}^{(1)}, \qquad
\boldsymbol{h} = \tanh(\boldsymbol{z}), \qquad
h_{\boldsymbol{\theta}}(x) = \boldsymbol{W}^{(2)}\boldsymbol{h} + b^{(2)},
$$
trained under the mean-squared-error loss $\hat R(\boldsymbol{\theta}) = \frac{1}{N}\sum_i (h_{\boldsymbol{\theta}}(x_i)-y_i)^2$ (Eq. 7.13, the Gaussian negative log-likelihood up to a constant).

Run the setup cell first, then work through the parts below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import shutil

fontsi, fontsi2 = 15, 20
if shutil.which('latex') is not None:
    plt.rc('text', usetex=True)
    plt.rcParams['font.serif'] = ['Computer Modern']
plt.rc('font', family='serif')
plt.rcParams.update({
    'font.size': fontsi, 'axes.titlesize': fontsi, 'axes.labelsize': fontsi2,
    'xtick.labelsize': fontsi, 'ytick.labelsize': fontsi,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': False, 'legend.frameon': False, 'figure.dpi': 130,
})

# ------------------------------------------------------------------
# Load the regression dataset. Each row is a pair (x_i, y_i). The
# data-generating process is deliberately NOT disclosed: your job is to
# fit it, not to look it up. The data carry additive observation noise,
# so the error cannot be driven below the noise variance (the irreducible
# sigma^2 of Eq. 7.10).
#   - sheet_9_data.txt : 200 TRAINING points -- fit the network on these
#   - sheet_9_test.txt : 2000 held-out TEST points -- ONLY for evaluation
# ------------------------------------------------------------------
train = np.loadtxt('./data/sheet_9_data.txt')
test  = np.loadtxt('./data/sheet_9_test.txt')
X,  y  = train[:, :1], train[:, 1:2]
Xt, yt = test[:, :1],  test[:, 1:2]

# standardise the inputs using the TRAINING statistics only
x_mu, x_sd = X.mean(0), X.std(0)
Xs  = (X  - x_mu) / x_sd
Xts = (Xt - x_mu) / x_sd
print(f"{X.shape[0]} training points,  {Xt.shape[0]} test points loaded")

a. Initialise the parameters $\boldsymbol{W}^{(1)}, \boldsymbol{b}^{(1)}, \boldsymbol{W}^{(2)}, b^{(2)}$ with small random weights (biases zero). Write a function that, given the parameters and the inputs, computes the network output *and* the gradient of the MSE loss with respect to every parameter by backpropagation. For this one-hidden-layer network show that the chain rule gives,
$$
\frac{\partial\hat R}{\partial \boldsymbol{W}^{(2)}} = \frac{2}{N}\sum_i r_i\,\boldsymbol{h}_i^\top,\qquad
\frac{\partial\hat R}{\partial b^{(2)}} = \frac{2}{N}\sum_i r_i,
$$
$$
\boldsymbol{\delta}_i = \big(\boldsymbol{W}^{(2)}\big)^\top r_i \odot \big(1-\tanh^2\boldsymbol{z}_i\big),\qquad
\frac{\partial\hat R}{\partial \boldsymbol{W}^{(1)}} = \frac{2}{N}\sum_i \boldsymbol{\delta}_i\,x_i,\qquad
\frac{\partial\hat R}{\partial \boldsymbol{b}^{(1)}} = \frac{2}{N}\sum_i \boldsymbol{\delta}_i,
$$
 with $r_i = h_{\boldsymbol{\theta}}(x_i)-y_i$ and where $a'(z)=1-\tanh^2 z$ is the derivative of the hidden activation.

Pick one weight, perturb it by $\pm\epsilon$ (with $\epsilon\approx10^{-6}$) and compare the central finite difference $[\hat R(\theta+\epsilon)-\hat R(\theta-\epsilon)]/(2\epsilon)$ to your analytic gradient. They should agree to $\sim$6 digits.

b. Train a network with $H=32$ hidden units using plain gradient descent (Eq. 7.26), $\boldsymbol{\theta}\leftarrow\boldsymbol{\theta}-\eta\,\nabla_{\boldsymbol{\theta}}\hat R$, where every step uses the training set. Record the loss at each iteration and plot it against the iteration number. Try a few learning rates $\eta$ and comment on what happens when $\eta$ is too small or too large.

c. Now replace the full-batch gradient by the mini-batch estimate of Eq. 7.30: at each step, draw a small random subset $\mathcal{B}$ of the data (e.g. $|\mathcal{B}|=16$) and use only those points to estimate the gradient. A full sweep through the data is one epoch. Train the same network and overlay its loss curve on the full-batch one (plot against epochs so the budgets are comparable). 

In both cases, also evaluate the MSE on the held-out test set `sheet_9_test.txt` once per epoch and plot it alongside the training MSE. How large is the gap between training and test error?

d. For your best run, plot the network prediction $h_{\boldsymbol{\theta}}(x)$ on a dense grid over $[-1,1]$ together with the training points. Report both the final training and test MSE. From your plot, what do you think the underlying function $h^\star(x)$ looks like?